# Preprocessing inputs
Let's mask potential credit card details and lower case the input before passing it to the LLM



In [ ]:
%pip install -qU langchain-ollama --quiet

Import libraries

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

Initializing the model

In [2]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Create a prompt to use with the chain

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Be concise"),
    ("user", "{input}")
    ])

Function to mask credit card numbers from a string

In [4]:
import re
def mask_credit_card_numbers(input_dict:dict) -> dict:
    """Function to mask credit card numbers from a string"""
    clean_input = re.sub(r'\b\d{4}[-.\s]\d{4}[-.\s]\d{4}\b', 'XXXX-XXXX-XXXX-XXXX', str(input_dict["input"]))
    return {"input": clean_input}

mask_credit_card_numbers({"input": "My card number is 1234-2345-3456-4567"})

{'input': 'My card number is XXXX-XXXX-XXXX-XXXX-4567'}

Function to lower case the input. Not necessary just practising LCEL chains

In [5]:
def lower_case(input_dict:dict) -> dict:
    """Function to lower case the input"""
    return {"input": input_dict["input"].lower()}

lower_case({"input":"ABC abc"})


{'input': 'abc abc'}

Create the chain using ```LCEL```

In [6]:
#chain = RunnableLambda(lower_case) | RunnableLambda(mask_credit_card_numbers) | prompt | llm | StrOutputParser()
preprocessor = RunnableLambda(lower_case) | RunnableLambda(mask_credit_card_numbers)
chain = preprocessor | prompt | llm | StrOutputParser()

In [11]:
# This shows the type of runnable objects in the chain
print("Number of steps in the chain: ", len(chain.steps))
for s in chain.steps:
    print(type(s))

Number of steps in the chain:  5
<class 'langchain_core.runnables.base.RunnableLambda'>
<class 'langchain_core.runnables.base.RunnableLambda'>
<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
<class 'langchain_ollama.chat_models.ChatOllama'>
<class 'langchain_core.output_parsers.string.StrOutputParser'>


Test the chain with "invoke" method

In [ ]:
response = chain.invoke({"input": "Please memorize this number 1234-2345-3456-4567. Can you print it back to me the exact same way I told you?"})

In [17]:
from IPython.display import display, Markdown
display(Markdown(response))

XXXX-XXXX-XXXX-XXXX-4567